Logic extracted to monitoring/pipeline.py

In [1]:
import sys
import os

PROJECT_ROOT = r"C:\Users\Sahil.s.Vernekar\OneDrive\Documents\ML\Churn\ml_system"
sys.path.insert(0, PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)

import json
import datetime
import random

Project root added: C:\Users\Sahil.s.Vernekar\OneDrive\Documents\ML\Churn\ml_system


In [2]:
from ml_db.mongo_client import temp_prediction_log_collection
N = 5
reference_data_cursor = temp_prediction_log_collection.find().sort("timestamp", 1).limit(N)
current_data_cursor = temp_prediction_log_collection.find().sort("timestamp", -1).limit(N)

reference_data = list(reference_data_cursor)
current_data   = list(current_data_cursor)

print(reference_data)

[{'_id': ObjectId('694fac434ff3c44f85fa4e16'), 'customer_id': 'C400', 'timestamp': '2025-12-27T09:52:03.621222', 'model_version': 'v1', 'features': {'tenure': 10, 'MonthlyCharges': 70.5, 'TotalCharges': 85.0, 'Contract': 'Month-to-month', 'PaymentMethod': 'Electronic check', 'InternetService': 'Fiber optic', 'num_support_tickets': 21}, 'prediction': 1.0, 'prediction_percentage': [0.6696590185165405], 'actual_label': 1}, {'_id': ObjectId('69520f6efd46dcdd9098dc9a'), 'customer_id': 'C401', 'timestamp': '2025-12-29T05:19:42.236005', 'model_version': 'v1', 'features': {'tenure': 3, 'MonthlyCharges': 89.5, 'TotalCharges': 268.5, 'Contract': 'Month-to-month', 'PaymentMethod': 'Electronic check', 'InternetService': 'Fiber optic', 'num_support_tickets': 9}, 'prediction': 1.0, 'prediction_percentage': [0.7100282311439514], 'actual_label': 1}, {'_id': ObjectId('69520f99fd46dcdd9098dc9b'), 'customer_id': 'C402', 'timestamp': '2025-12-29T05:20:25.138213', 'model_version': 'v1', 'features': {'tenur

In [3]:
from monitoring.performance_utility import compute_confusion_matrix
ref_metrics = compute_confusion_matrix(reference_data)
curr_metrics = compute_confusion_matrix(current_data)

Project root added: C:\Users\Sahil.s.Vernekar\OneDrive\Documents\ML\Churn\ml_system


In [4]:
print(ref_metrics)
print(curr_metrics)

{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [5]:
combined_metrics = {}
accuracy_drop = ref_metrics['accuracy'] - curr_metrics['accuracy']
f1_drop = ref_metrics['f1'] - curr_metrics['f1']
metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    combined_metrics[i] = {
        'reference': ref_metrics[i],
        'current': curr_metrics[i]
    }

combined_metrics['accuracy_drop'] = accuracy_drop
combined_metrics['f1_drop'] = f1_drop


In [6]:
print(combined_metrics)

{'accuracy': {'reference': 1.0, 'current': 1.0}, 'precision': {'reference': 1.0, 'current': 1.0}, 'recall': {'reference': 1.0, 'current': 1.0}, 'f1': {'reference': 1.0, 'current': 1.0}, 'accuracy_drop': 0.0, 'f1_drop': 0.0}


In [7]:
from monitoring.performance_utility import drift_decision

drift_status = drift_decision(accuracy_drop, f1_drop)
print(drift_status)

no_drift


In [9]:
from datetime import datetime
from ml_db.mongo_client import performance_drift_reports_collection

performance_drift_report = {
    "drift_type": "performance_drift",
    "model_version": "v1",
    
    "window": {
        "type": "time_based",
        "size": 5,
        "label_source": "simulated"
  },

    "reference_metrics": ref_metrics,
    "current_metrics": curr_metrics,
    
    "metric_drops": {
        "accuracy_drop": accuracy_drop,
        "f1_drop": curr_metrics
  },

    "status": drift_status,
    "generated_at": datetime.utcnow().isoformat()
}

performance_drift_reports_collection.insert_one(performance_drift_report)


InsertOneResult(ObjectId('6953994e71aa82511e8572f0'), acknowledged=True)